# Vehicle Registration Metrics: CFAR & FMI

1. **CFAR (Clean Fuel Adoption Rate)** — % of registered vehicles using clean fuel types (EV, CNG, Hybrid), measuring the pace of green energy transition in transport.
2. **FMI (Fleet Modernisation Index)** — % of registered vehicles that are both emission-compliant (BS6, the strictest norm in this dataset) **and** relatively new (age ≤ 5 years), measuring how modern/low-risk the active fleet is.

For each metric we produce:
- A single **overall KPI** value (headline stat)
- A breakdown **by State** (for bar chart ranking / comparison)
- A breakdown **by Registration_Year** (for trend line over time)
- A breakdown **by State + Vehicle_Category** (for heatmap-style analysis)

These grouped tables are kept separate from the row-level dataframe since visualization tools work best with compact "one row per data point to plot" tables, rather than a fully merged row-level dataframe with repeated values.

## Step 0: Load the data

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/cleaned_dvs_data.csv")
df.head()

,Registration_Number,Registration_Date,Registration_Year,State,RTO_Office,Vehicle_Category,Vehicle_Sub_Type,Manufacturer_Brand,Fuel_Type,Emission_Norm,Engine_CC,Seating_Capacity,Vehicle_Age_Years
0,REG100001,13-12-2019,2019,Madhya Pradesh,Bhopal (MP-04),2W,Scooter,Bajaj,Petrol,BS4,400.0,2,5
1,REG100002,14-01-2020,2020,Karnataka,Belgaum (KA-22),2W,Motorcycle,Honda,Petrol,BS6,400.0,2,4
2,REG100003,23-03-2023,2023,Karnataka,Delhi South (DL-09),2W,Scooter,Ola Electric,Petrol,BS6,250.0,2,1
3,REG100004,26-06-2021,2021,Karnataka,Kota (RJ-20),2W,Scooter,Honda,Petrol,BS6,160.0,2,3
4,REG100005,13-11-2022,2022,Karnataka,Ahmedabad (GJ-01),4W,Hatchback,Honda,Petrol,BS6,1598.0,5,2


## Step 0b: Clean `Emission_Norm` for EVs

`Emission_Norm` (BS3/BS4/BS6) is a combustion tailpipe-emission standard, but Vahan auto-attaches a BSx value to EV records too, even though EVs have no tailpipe. Since that value is a data artifact rather than a real measurement, we relabel `Fuel_Type == 'EV'` rows to `'ZEV'` (Zero Emission Vehicle) in a new cleaned column, keeping the original `Emission_Norm` untouched for auditability. Hybrid vehicles keep their real BS value since they still have a genuine combustion engine and tailpipe.

In [19]:
# Diagnostic: see what BSx value Vahan auto-assigns to EV records
print(df[df['Fuel_Type'] == 'EV']['Emission_Norm'].value_counts(dropna=False))

# Derive a cleaned column instead of overwriting the raw Emission_Norm column,
# so the original Vahan-reported value stays visible/auditable.
df['Emission_Norm_Clean'] = df['Emission_Norm'].where(df['Fuel_Type'] != 'EV', 'ZEV')

df['Emission_Norm_Clean'].value_counts(dropna=False)

Emission_Norm
BS6    638
BS4    110
Name: count, dtype: int64


,count
Emission_Norm_Clean,
BS6,2449
ZEV,748
BS4,496
BS3,7


## Metric A: Clean Fuel Adoption Rate (CFAR)

**Formula:**

$$CFAR = \frac{\text{Count(Fuel\_Type} \in \{\text{'EV', 'CNG', 'Hybrid'}\})}{\text{Total registrations}} \times 100$$

In [20]:
clean_fuel_types = ['EV', 'CNG', 'Hybrid']

# Binary flag: True if this vehicle's fuel type counts as "clean"
df['is_clean'] = df['Fuel_Type'].isin(clean_fuel_types)

# --- Overall KPI: single headline number across the whole dataset ---
overall_cfar = df['is_clean'].mean() * 100
print(f"Overall CFAR: {overall_cfar:.2f}%")

Overall CFAR: 37.35%


In [21]:
# --- By State: which states lead/lag in clean fuel adoption ---
# Useful for a horizontal bar chart ranking states.
cfar_by_state = (
    df.groupby('State')['is_clean'].mean() * 100
).reset_index(name='CFAR').sort_values('CFAR', ascending=False)

cfar_by_state

,State,CFAR
1,Gujarat,42.561983
4,Madhya Pradesh,41.984733
5,Maharashtra,39.555556
2,Karnataka,38.051282
7,Rajasthan,37.551020
11,West Bengal,37.394958
8,Tamil Nadu,37.307692
9,Telangana,35.042735
3,Kerala,34.944238
6,Punjab,34.615385


In [22]:
# --- By Year: is clean fuel adoption improving over time? ---
# Useful for a line chart trend.
cfar_by_year = (
    df.groupby('Registration_Year')['is_clean'].mean() * 100
).reset_index(name='CFAR').sort_values('Registration_Year')

cfar_by_year

,Registration_Year,CFAR
0,2018,36.407767
1,2019,35.179153
2,2020,37.471783
3,2021,37.815126
4,2022,40.000000
5,2023,35.404455
6,2024,37.883959


In [23]:
# --- By State + Vehicle_Category: which state-category combo lags most ---
# Useful for a heatmap (State on one axis, Vehicle_Category on the other).
cfar_by_state_category = (
    df.groupby(['State', 'Vehicle_Category'])['is_clean'].mean() * 100
).reset_index(name='CFAR')

cfar_by_state_category

,State,Vehicle_Category,CFAR
0,Delhi,2W,37.500000
1,Delhi,3W,87.500000
2,Delhi,4W,24.000000
3,Delhi,HCV,20.000000
4,Delhi,LCV,30.000000
...,...,...,...
67,West Bengal,3W,75.000000
68,West Bengal,4W,25.675676
69,West Bengal,HCV,20.000000
70,West Bengal,LCV,42.857143


## Metric B: Fleet Modernisation Index (FMI)

**Formula:**

$$FMI = \frac{\text{Count(Emission\_Norm\_Clean} \in \{\text{'BS6', 'ZEV'}\} \cap \text{Vehicle\_Age\_Years} \le 5)}{\text{Total registrations}} \times 100$$

> Note: the dataset only contains `{BS3, BS4, BS6}` as emission norms, but Vahan auto-attaches a BSx placeholder to EV records even though EVs have no tailpipe. We treat `'ZEV'` (the relabeled EV rows, see Step 0b) as compliant alongside `'BS6'`, since a zero-emission vehicle is at least as "modern/low-risk" as the strictest combustion standard. BS4/BS3 combustion vehicles remain excluded.

In [24]:
compliant_emission_norms = ['BS6', 'ZEV']
max_age = 5

# Binary flag: True if vehicle meets BOTH conditions (compound AND, not OR)
df['is_compliant'] = (
    df['Emission_Norm_Clean'].isin(compliant_emission_norms) &
    (df['Vehicle_Age_Years'] <= max_age)
)

print(df['is_compliant'].sum())
# --- Overall KPI: single headline number across the whole dataset ---
overall_fmi = df['is_compliant'].mean() * 100
print(f"Overall FMI: {overall_fmi:.2f}%")

3160
Overall FMI: 85.41%


In [25]:
# --- By State: which states have the most modernised/compliant fleets ---
fmi_by_state = (
    df.groupby('State')['is_compliant'].mean() * 100
).reset_index(name='FMI').sort_values('FMI', ascending=False)

fmi_by_state

,State,FMI
5,Maharashtra,87.111111
8,Tamil Nadu,86.923077
1,Gujarat,86.776860
9,Telangana,86.752137
3,Kerala,86.617100
11,West Bengal,86.134454
7,Rajasthan,86.122449
2,Karnataka,85.333333
0,Delhi,85.200000
6,Punjab,84.615385


In [26]:
# --- By Year: is the fleet modernising over time? ---
fmi_by_year = (
    df.groupby('Registration_Year')['is_compliant'].mean() * 100
).reset_index(name='FMI').sort_values('Registration_Year')

fmi_by_year

,Registration_Year,FMI
0,2018,0.000000
1,2019,25.732899
2,2020,87.133183
3,2021,95.798319
4,2022,98.169014
5,2023,98.710434
6,2024,100.000000


In [27]:
# --- By State + Vehicle_Category: pinpoint which segment needs scrappage
#     schemes or targeted policy intervention ---
fmi_by_state_category = (
    df.groupby(['State', 'Vehicle_Category'])['is_compliant'].mean() * 100
).reset_index(name='FMI')

fmi_by_state_category

,State,Vehicle_Category,FMI
0,Delhi,2W,83.593750
1,Delhi,3W,93.750000
2,Delhi,4W,88.000000
3,Delhi,HCV,60.000000
4,Delhi,LCV,100.000000
...,...,...,...
67,West Bengal,3W,100.000000
68,West Bengal,4W,83.783784
69,West Bengal,HCV,100.000000
70,West Bengal,LCV,85.714286


## Summary

In [28]:
print("Overall CFAR (Clean Fuel Adoption Rate):", round(overall_cfar, 2), "%")
print("Overall FMI  (Fleet Modernisation Index):", round(overall_fmi, 2), "%")

Overall CFAR (Clean Fuel Adoption Rate): 37.35 %
Overall FMI  (Fleet Modernisation Index): 85.41 %


## Row-Level Merge (for Dashboard Use)

Here we merge `CFAR` and `FMI` back onto the full row-level `df`, grouped by **Registration_Year + State** (matching the metrics' natural time + spatial dimensions, `t` and `d`).

In [29]:
# --- Compute CFAR and FMI per (Registration_Year, State) group ---
dashboard_group_cols = ['Registration_Year', 'State']

cfar_fmi_lookup = df.groupby(dashboard_group_cols).agg(
    clean_count=('is_clean', 'sum'),
    compliant_count=('is_compliant', 'sum'),
    total_count=('Registration_Number', 'count')
).reset_index()

cfar_fmi_lookup['CFAR'] = (cfar_fmi_lookup['clean_count'] / cfar_fmi_lookup['total_count']) * 100
cfar_fmi_lookup['FMI'] = (cfar_fmi_lookup['compliant_count'] / cfar_fmi_lookup['total_count']) * 100

# --- Merge both metrics back onto the full row-level dataframe ---
df = df.merge(
    cfar_fmi_lookup[dashboard_group_cols + ['CFAR', 'FMI']],
    on=dashboard_group_cols,
    how='left'
)


In [30]:
df['CFAR'].unique()

array([44.44444444, 37.5       , 37.17948718, 32.88590604, 43.33333333,
       41.29032258, 31.3253012 , 40.74074074, 50.        , 44.11764706,
       35.71428571, 39.13043478, 33.33333333, 23.80952381, 55.        ,
       52.63157895, 30.76923077, 27.02702703, 32.60869565, 41.66666667,
       34.28571429, 32.72727273, 36.36363636, 34.61538462, 48.8372093 ,
       32.8358209 , 25.71428571, 30.        , 36.17021277, 29.03225806,
       48.14814815, 38.88888889, 36.66666667, 30.50847458, 37.25490196,
       39.21568627, 42.59259259, 31.25      , 26.66666667, 45.16129032,
       39.53488372, 32.25806452, 45.45454545, 34.7826087 , 36.84210526,
       41.02564103, 45.71428571, 42.85714286, 44.89795918, 27.77777778,
       28.57142857, 38.46153846, 27.27272727, 42.10526316, 17.64705882,
       41.50943396, 35.29411765, 31.81818182, 21.62162162, 18.18181818,
       38.70967742, 52.        , 38.23529412, 44.73684211, 54.54545455,
       41.86046512, 45.        , 20.        ])

In [31]:
df.to_csv('cleaned_dvs_data_with_cfar_fmi.csv', index=False)